# SOFC Dataset Analysis

This notebook demonstrates how to analyze the generated SOFC simulation dataset.

In [ ]:
import sys
import os
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from visualization.visualizer import SOFCVisualizer

# Set up plotting
plt.style.use('seaborn-v0_8')
%matplotlib inline

## Load Dataset

In [ ]:
# Initialize visualizer
dataset_dir = "../data/sofc_dataset"  # Update path as needed
visualizer = SOFCVisualizer(dataset_dir)

print(f"Dataset loaded from: {dataset_dir}")
if visualizer.parameter_df is not None:
    print(f"Number of samples: {len(visualizer.parameter_df)}")
    print(f"Number of parameters: {len(visualizer.parameter_df.columns) - 1}")
else:
    print("No parameter data found")

## Parameter Analysis

In [ ]:
# Plot parameter distributions
if visualizer.parameter_df is not None:
    fig = visualizer.plot_parameter_distributions()
    plt.show()
else:
    print("Parameter data not available")

In [ ]:
# Plot parameter correlations
if visualizer.parameter_df is not None:
    fig = visualizer.plot_parameter_correlations()
    plt.show()
else:
    print("Parameter data not available")

## Performance Metrics Analysis

In [ ]:
# Plot performance metrics
if visualizer.metric_df is not None:
    fig = visualizer.plot_performance_metrics()
    plt.show()
    
    # Show metric statistics
    print("\nMetric Statistics:")
    metric_cols = [col for col in visualizer.metric_df.columns if col != 'sample_id']
    print(visualizer.metric_df[metric_cols].describe())
else:
    print("Metric data not available")

## Sensitivity Analysis

In [ ]:
# Perform sensitivity analysis for key metrics
if visualizer.metric_df is not None:
    key_metrics = [
        'electrochemical_average_current_density',
        'electrochemical_power_density',
        'thermal_average_temperature'
    ]
    
    available_metrics = [m for m in key_metrics if m in visualizer.metric_df.columns]
    
    for metric in available_metrics[:2]:  # Show first 2 available metrics
        try:
            fig = visualizer.plot_parameter_sensitivity(metric)
            plt.show()
        except Exception as e:
            print(f"Error plotting sensitivity for {metric}: {e}")
else:
    print("Metric data not available for sensitivity analysis")

## Interactive Dashboard

In [ ]:
# Create interactive dashboard
try:
    dashboard_html = visualizer.create_interactive_dashboard()
    
    # Save to file
    with open('interactive_dashboard.html', 'w') as f:
        f.write(dashboard_html)
    
    print("Interactive dashboard created: interactive_dashboard.html")
    print("Open this file in a web browser to view the interactive plots.")
    
except Exception as e:
    print(f"Error creating dashboard: {e}")

## Dataset Quality Assessment

In [ ]:
# Check for missing values and outliers
if visualizer.parameter_df is not None:
    print("Parameter Data Quality:")
    print(f"Missing values: {visualizer.parameter_df.isnull().sum().sum()}")
    print(f"Duplicate rows: {visualizer.parameter_df.duplicated().sum()}")
    
if visualizer.metric_df is not None:
    print("\nMetric Data Quality:")
    print(f"Missing values: {visualizer.metric_df.isnull().sum().sum()}")
    
    # Check for infinite values
    numeric_cols = visualizer.metric_df.select_dtypes(include=[np.number]).columns
    inf_count = np.isinf(visualizer.metric_df[numeric_cols]).sum().sum()
    print(f"Infinite values: {inf_count}")
    
    # Detect outliers using IQR method
    outlier_counts = {}
    for col in numeric_cols:
        if col != 'sample_id':
            Q1 = visualizer.metric_df[col].quantile(0.25)
            Q3 = visualizer.metric_df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            outliers = ((visualizer.metric_df[col] < lower_bound) | 
                       (visualizer.metric_df[col] > upper_bound)).sum()
            outlier_counts[col] = outliers
    
    print("\nOutliers (IQR method):")
    for col, count in outlier_counts.items():
        if count > 0:
            print(f"  {col}: {count} outliers")

## Generate Complete Analysis Report

In [ ]:
# Generate comprehensive analysis report
try:
    report_path = visualizer.generate_analysis_report()
    print(f"Analysis report generated: {report_path}")
    
    # Display report content
    with open(report_path, 'r') as f:
        report_content = f.read()
    
    print("\nReport Preview:")
    print(report_content[:1000] + "..." if len(report_content) > 1000 else report_content)
    
except Exception as e:
    print(f"Error generating report: {e}")